In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 16:15:55 WARN Utils: Your hostname, DESKTOP-0ETQBA1, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/09 16:15:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 16:16:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'4.1.1'

In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 16:16:05--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.225.31.10, 13.225.31.100, 13.225.31.154, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.225.31.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67.84M  36.5MB/s    in 1.9s    

2026-03-09 16:16:07 (36.5 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]



In [5]:
!ls -lh yellow_tripdata_2025-11.parquet

-rwxrwxrwx 1 home home 68M Dec 19 17:51 yellow_tripdata_2025-11.parquet


In [6]:
# df = spark.read \
#     .option("header", "true") \
#     .schema(yellow_schema) \
#     .parquet('yellow_tripdata_2025-11.parquet')

In [7]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [8]:
df = df.repartition(4)

In [9]:
df.write.mode("overwrite").parquet('data/pq/yellow/2025/11/')

In [10]:
!ls -lh ./data/pq/yellow/2025/11

total 98M
-rwxrwxrwx 1 home home   0 Mar  9 16:16 _SUCCESS
-rwxrwxrwx 1 home home 25M Mar  9 16:16 part-00000-ab309dc7-ad10-4448-aaf3-a93637ab3260-c000.snappy.parquet
-rwxrwxrwx 1 home home 25M Mar  9 16:16 part-00001-ab309dc7-ad10-4448-aaf3-a93637ab3260-c000.snappy.parquet
-rwxrwxrwx 1 home home 25M Mar  9 16:16 part-00002-ab309dc7-ad10-4448-aaf3-a93637ab3260-c000.snappy.parquet
-rwxrwxrwx 1 home home 25M Mar  9 16:16 part-00003-ab309dc7-ad10-4448-aaf3-a93637ab3260-c000.snappy.parquet


In [11]:
df = spark.read.parquet('data/pq/yellow/2025/11/')

## **Q3**: How many taxi trips were there on the 15th of November?

In [12]:
from pyspark.sql import functions as F

In [13]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [14]:
df \
    .withColumn('tpep_pickup_datetime', F.to_date(df.tpep_pickup_datetime)) \
    .filter("tpep_pickup_datetime = '2025-11-15'") \
    .count()

162604

In [15]:
df.filter(F.to_date(df.tpep_pickup_datetime) == '2025-11-15').count()

162604

In [16]:
df.registerTempTable('yellow_2025_11')

/mnt/c/Users/home/Desktop/de-zoomcamp/cohorts/2026/06-batch/.venv/lib/python3.14/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [17]:
spark.sql("""
SELECT
    COUNT(1)
FROM 
    yellow_2025_11
WHERE
    to_date(tpep_pickup_datetime) = '2025-11-15';
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



## **Q4**: What is the length of the longest trip in the dataset in hours?

In [18]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [19]:
# df \
#     .withColumn('duration', df.tpep_dropoff_datetime.cast('long') - df.tpep_pickup_datetime.cast('long')) \
#     .withColumn('tpep_pickup_datetime', F.to_date(df.tpep_pickup_datetime)) \
#     .groupBy('tpep_pickup_datetime') \
#         .max('duration') \
#     .orderBy('max(duration)', ascending=False) \
#     .limit(5) \
#     .show()

In [20]:
df.withColumn('duration_hours', 
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
).agg(F.max('duration_hours')).show()

+-------------------+
|max(duration_hours)|
+-------------------+
|  90.64666666666666|
+-------------------+



In [21]:
spark.sql("""
SELECT
    to_date(tpep_pickup_datetime) AS pickup_date,
    MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) AS duration_hours
FROM 
    yellow_2025_11
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 10;
""").show()

[Stage 17:=============================================>          (13 + 3) / 16]

+-----------+------------------+
|pickup_date|    duration_hours|
+-----------+------------------+
| 2025-11-26| 90.64666666666666|
| 2025-11-27| 76.94833333333334|
| 2025-11-03| 76.21388888888889|
| 2025-11-07| 69.28861111111111|
| 2025-11-18| 67.08055555555555|
| 2025-11-22| 63.36833333333333|
| 2025-11-01|56.382222222222225|
| 2025-11-05|42.720555555555556|
| 2025-11-06|41.614444444444445|
| 2025-11-24|38.074444444444445|
+-----------+------------------+



## **Q6**: Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [22]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 16:16:28--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.225.31.154, 13.225.31.100, 13.225.31.10, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.225.31.154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.3’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2026-03-09 16:16:28 (8.44 MB/s) - ‘taxi_zone_lookup.csv.3’ saved [12331/12331]



In [23]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [30]:
df.join(df_zones, df.PULocationID == df_zones.LocationID) \
    .groupBy('Zone') \
    .count() \
    .orderBy('count', ascending=True) \
    .limit(10) \
    .show(truncate=False)

+---------------------------------------------+-----+
|Zone                                         |count|
+---------------------------------------------+-----+
|Eltingville/Annadale/Prince's Bay            |1    |
|Governor's Island/Ellis Island/Liberty Island|1    |
|Arden Heights                                |1    |
|Port Richmond                                |3    |
|Rikers Island                                |4    |
|Rossville/Woodrow                            |4    |
|Great Kills                                  |4    |
|Green-Wood Cemetery                          |4    |
|Jamaica Bay                                  |5    |
|Westerleigh                                  |12   |
+---------------------------------------------+-----+



**Optional**: Most common locations pair

In [25]:
df_zones = spark.read.parquet('zones')

In [26]:
df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [27]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [28]:
df_zones.registerTempTable('zones')

In [29]:
spark.sql("""
SELECT
    CONCAT(pul.Zone, ' / ', dol.Zone) AS pu_do_pair,
    COUNT(1)
FROM 
    yellow_2025_11 yellow 
    LEFT JOIN zones pul ON yellow.PULocationID = pul.LocationID
    LEFT JOIN zones dol ON yellow.DOLocationID = dol.LocationID
GROUP BY 
    1
ORDER BY
    2 DESC
LIMIT 5;
""").show(truncate=False)

[Stage 27:==========================================>             (12 + 4) / 16]

+---------------------------------------------+--------+
|pu_do_pair                                   |count(1)|
+---------------------------------------------+--------+
|Upper East Side South / Upper East Side North|28326   |
|Upper East Side North / Upper East Side South|24720   |
|Upper East Side South / Upper East Side South|20512   |
|Upper East Side North / Upper East Side North|17820   |
|Midtown Center / Upper East Side South       |13444   |
+---------------------------------------------+--------+

